In [1]:
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import h5py
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from einops import rearrange
from more_itertools import one
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints")

jax.config.update("jax_default_device", jax.devices("gpu")[7])

In [2]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

In [ ]:
dataset_test = xr.open_dataset(
    datadir
    / "datasets/shallow_water_1d/data/test/shallow_water_1d_test_grf_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir
            / "shallow_water_1d"
            / model_type
            / "OneStepLoss"
            / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=1500,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
import json


with open("../../data/analysis/shallow_water_indistribution.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(model_name, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")

## Long term

In [4]:
@eqx.filter_jit
def metrics_long_rollout(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]
    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]
    l2 = jax.vmap(relative_L2_error)(u_pred, u_data)
    l_inf = jax.vmap(relative_L_infty_error)(u_pred, u_data)
    return {"l2": l2, "l_inf": l_inf}

In [ ]:
results_dict = defaultdict(list)
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir
            / "shallow_water_1d"
            / model_type
            / "OneStepLoss"
            / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: metrics_long_rollout(model, u, (dt, dx))),
            values,
            batch_size=1000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(results)

 33%|███▎      | 1/3 [00:25<00:51, 25.58s/it]

{'l2': (8952, 81), 'l_inf': (8952, 81)}


 67%|██████▋   | 2/3 [00:40<00:19, 19.11s/it]

{'l2': (8952, 81), 'l_inf': (8952, 81)}


100%|██████████| 3/3 [00:54<00:00, 18.29s/it]


{'l2': (8952, 81), 'l_inf': (8952, 81)}


... (8 outputs left out)

 67%|██████▋   | 2/3 [06:47<03:17, 197.83s/it]

{'l2': (8952, 81), 'l_inf': (8952, 81)}


100%|██████████| 3/3 [09:43<00:00, 194.37s/it]

{'l2': (8952, 81), 'l_inf': (8952, 81)}


In [8]:
results_stacked = {
    k: jax.tree.transpose(
        jax.tree.structure(["*"] * 3),
        None,
        v,
    )
    for k, v in results_dict.items()
}
results_stacked = jax.tree.map(
    np.stack, results_stacked, is_leaf=lambda x: isinstance(x, list)
)

In [9]:
with h5py.File("../../data/analysis/shallow_water_1d_long_rollout.hdf5", "w") as f:
    for model_name, res_dict in results_stacked.items():
        g = f.create_group(model_name)
        for metric_name, val in res_dict.items():
            g.create_dataset(metric_name, data=val)

## OOD

In [3]:
dataset_test = xr.open_dataset(
    datadir
    / "datasets/shallow_water_1d/data/test/shallow_water_1d_test_step_v2_grf_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

(608158, 40, 2, 100)

In [ ]:
import matplotlib.pyplot as plt


fig, axes = plt.subplots(2, 5, figsize=(10, 5), sharey="row")
for i in range(axes.shape[1]):
    axes[0, i].plot(dataset_test.coords["x"], dataset_test["values"][0, 4 * i, 0])
    axes[1, i].plot(dataset_test.coords["x"], dataset_test["values"][0, 4 * i, 1])

In [4]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir
            / "shallow_water_1d"
            / model_type
            / "OneStepLoss"
            / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=1500,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [03:09<06:18, 189.46s/it]

{'l2_onestep': (608158,), 'l2_rollout': (608158,), 'l_inf_onestep': (608158,), 'l_inf_rollout': (608158,)}


 67%|██████▋   | 2/3 [06:07<03:02, 182.79s/it]

{'l2_onestep': (608158,), 'l2_rollout': (608158,), 'l_inf_onestep': (608158,), 'l_inf_rollout': (608158,)}


100%|██████████| 3/3 [09:02<00:00, 180.94s/it]


... (9 outputs left out)

 67%|██████▋   | 2/3 [1:00:15<30:07, 1807.12s/it]

{'l2_onestep': (608158,), 'l2_rollout': (608158,), 'l_inf_onestep': (608158,), 'l_inf_rollout': (608158,)}


100%|██████████| 3/3 [1:29:38<00:00, 1792.69s/it]

{'l2_onestep': (608158,), 'l2_rollout': (608158,), 'l_inf_onestep': (608158,), 'l_inf_rollout': (608158,)}


In [5]:
import json


with open("../../data/analysis/shallow_water_ood_step_grf_v2.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [6]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(model_name, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")

DPOT l2_onestep: 5.95e-03±1.87e-04
DPOT l2_rollout: 7.54e-02±1.32e-03
DPOT l_inf_onestep: 2.05e-02±6.32e-04
DPOT l_inf_rollout: 3.30e-01±3.80e-03
HyperFluxFNOLocal l2_onestep: 6.46e-03±7.65e-05
HyperFluxFNOLocal l2_rollout: 6.41e-02±2.72e-04
HyperFluxFNOLocal l_inf_onestep: 2.35e-02±1.80e-05
HyperFluxFNOLocal l_inf_rollout: 3.16e-01±7.21e-04
DISCO l2_onestep: 1.30e-02±5.42e-04
DISCO l2_rollout: 1.04e-01±4.36e-03
DISCO l_inf_onestep: 4.69e-02±2.21e-03
DISCO l_inf_rollout: 3.87e-01±1.17e-02
